# NLU HOSTAGE — ai_1_nlu_v2_200

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [1]:
from pathlib import Path
DATASET_FILENAME = "v2_chat_dataset_200.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v2_200"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


Dataset aktif: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v2_200\data\v2_chat_dataset_200.csv


In [2]:
# MODUL BERSAMA: EDA + SVM + SVM TUNING + NB + NB TUNING + TRANSFORMER
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=4):
    return train_transformer(DATASET_PATH, MODEL_DIR, epochs=epochs, device=TRANSFORMER_DEVICE)

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("accusing", "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),
    ("defending", "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),
    ("bluffing", "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),
    ("probing", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),
    ("deflecting", "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),
    ("persuading", "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),
    ("claiming", "Klaim gw Civilian, gw gak punya skill malam."),
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),
train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().
Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.
Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.


In [3]:
# JALANKAN SEMUA MODEL: empat model CPU, lalu Transformer CUDA dan 10 chat uji.
RUN_TRANSFORMER = True
RUN_TUNING = False
TRANSFORMER_EPOCHS = 4
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)



MENJALANKAN: SVM baseline
SVM | train=1292 | test=323 | kelas=8



--- Evaluasi SVM baseline (holdout test set) ---
Accuracy    : 0.6966
Macro F1    : 0.6955
Weighted F1 : 0.6935
              precision    recall  f1-score   support

    accusing       0.61      0.56      0.58        41
    bluffing       0.56      0.49      0.52        41
    claiming       0.65      0.65      0.65        40
   defending       0.65      0.70      0.67        40
  deflecting       0.72      0.76      0.74        41
     neutral       0.95      0.92      0.93        38
  persuading       0.60      0.59      0.59        41
     probing       0.83      0.93      0.87        41

    accuracy                           0.70       323
   macro avg       0.69      0.70      0.70       323
weighted avg       0.69      0.70      0.69       323

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v2_200\models\intent_classifier_svm.pkl

MENJALANKAN: Naive Bayes baseline
Naive Bayes | train=1292 | test=323 | kelas=8

--- Evaluasi Naive Bayes baseline 

C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Map:   0%|          | 0/1292 [00:00<?, ? examples/s]


Map: 100%|██████████| 1292/1292 [00:00<00:00, 6700.49 examples/s]

...

expected=defending    | predicted=neutral      | confidence= 69.94% | MISS

--- 10 chat uji: Naive Bayes baseline ---
expected=accusing     | predicted=accusing     | confidence= 40.87% | OK
expected=defending    | predicted=defending    | confidence= 54.72% | OK
expected=bluffing     | predicted=bluffing     | confidence= 52.04% | OK
expected=probing      | predicted=probing      | confidence= 75.09% | OK
expected=deflecting   | predicted=deflecting   | confidence= 29.80% | OK
expected=persuading   | predicted=accusing     | confidence= 27.60% | MISS
expected=claiming     | predicted=claiming     | confidence= 50.23% | OK
expected=neutral      | predicted=neutral      | confidence= 38.24% | OK
expected=accusing     | predicted=deflecting   | confidence= 28.25% | MISS
expected=defending    | predicted=neutral      | confidence= 43.46% | MISS

--- 10 chat uji: IndoBERT Transformer ---


expected=accusing     | predicted=persuading   | confidence= 74.87% | MISS
expected=defending    | predicted=deflecting   | confidence= 66.89% | MISS
expected=bluffing     | predicted=bluffing     | confidence= 41.98% | OK
expected=probing      | predicted=probing      | confidence= 98.41% | OK
expected=deflecting   | predicted=deflecting   | confidence= 64.60% | OK
expected=persuading   | predicted=persuading   | confidence= 88.97% | OK
expected=claiming     | predicted=deflecting   | confidence= 42.63% | MISS
expected=neutral      | predicted=probing      | confidence= 31.86% | MISS
expected=accusing     | predicted=persuading   | confidence= 73.80% | MISS
expected=defending    | predicted=deflecting   | confidence= 73.62% | MISS
RINGKASAN EVALUASI HOLDOUT:


,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,waktu_detik,status
0,IndoBERT Transformer,0.8514,0.8509,0.8501,49.8,berhasil
1,SVM baseline,0.6966,0.6955,0.6935,0.2,berhasil
2,Naive Bayes baseline,0.6904,0.6934,0.6920,0.2,berhasil


RINGKASAN 10 CHAT UJI:


,model,benar_dari_10,akurasi_10_chat
0,SVM baseline,7,0.7
1,Naive Bayes baseline,7,0.7
2,IndoBERT Transformer,4,0.4


## Laporan eksekusi notebook

Tuning SVM dan Naive Bayes dilewati untuk mempercepat run ini. Output training lengkap tersimpan pada cell tepat di atas.

| Model | Macro-F1 holdout | Uji 10 chat |
|---|---:|---:|
| SVM baseline | 0.6955 | 7/10 |
| Naive Bayes baseline | 0.6934 | 7/10 |
| IndoBERT Transformer (GPU) | 0.8509 | 4/10 |

Transformer unggul pada holdout; SVM dan Naive Bayes lebih konsisten pada 10 chat manual.